In [33]:
import napari
import tifffile
import numpy as np
from pathlib import Path
from magicgui import magicgui
from magicgui.widgets import TextEdit, PushButton, Container, ComboBox
from napari.qt.threading import thread_worker
import os
import tempfile
import numpy as np
from scipy.ndimage import distance_transform_edt

import matplotlib.pyplot as plt


In [34]:

def safe_imwrite(path, arr, *, imagej=True, resolution=None, resolutionunit=None, metadata=None):
    path = os.fspath(path)
    arr = np.asarray(arr)

    # ImageJ TIFF only supports these dtypes -> fail loudly BEFORE touching disk.
    # if imagej and arr.dtype not in (np.uint8, np.uint16, np.int16, np.float32):
    #     raise TypeError(
    #         f"imagej=True cannot store dtype {arr.dtype}. Cast first, e.g. "
    #         f"arr.astype(np.float32) (intensities) or arr.astype(np.uint16) (labels)."
    #     )

    folder = os.path.dirname(path) or "."
    fd, tmp = tempfile.mkstemp(suffix=".tmp.tif", dir=folder)
    os.close(fd)
    try:
        kwargs = {"imagej": imagej}
        if resolution is not None:
            kwargs["resolution"] = resolution
        if resolutionunit is not None:
            kwargs["resolutionunit"] = resolutionunit
        if metadata is not None:
            kwargs["metadata"] = metadata

        tifffile.imwrite(tmp, arr, **kwargs)

        # Verify the temp file matches the source before replacing it, using a
        # memory-map + per-slice comparison so we never allocate a whole extra
        # copy or a whole boolean array.
        written = tifffile.imread(tmp, out="memmap")
        try:
            ok = tuple(written.shape) == tuple(arr.shape)
            if ok and arr.ndim == 0:
                ok = bool(np.array_equal(written, arr))
            elif ok:
                for i in range(arr.shape[0]):
                    if not np.array_equal(written[i], arr[i]):
                        ok = False
                        break
        finally:
            # Close the memory-map's file handle before replace/remove, otherwise
            # os.replace/os.remove fails on Windows ("file in use").
            mm = getattr(written, "_mmap", None)
            if mm is not None:
                mm.close()
            del written
        if not ok:
            raise ValueError("verification failed: written data != source array")

        os.replace(tmp, path)  # atomic on the same filesystem
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)
    return path


In [35]:
def read_image_and_meta(path):
    """Read a tif as a (Z, C, Y, X) array along with the metadata needed to
    write it back out unchanged (pixel size / spacing / unit)."""
    with tifffile.TiffFile(path) as tif:
        series = max(tif.series, key=lambda s: s.size)
        arr = series.asarray()
        axes = series.axes
        ij_meta = dict(tif.imagej_metadata or {})

        def _rational(tag):
            if tag is None:
                return None
            val = tag.value
            if isinstance(val, tuple) and len(val) == 2:
                return val[0] / val[1] if val[1] else None
            return val

        xres = yres = resunit = None
        for page in tif.pages:
            if "XResolution" in page.tags:
                xres = _rational(page.tags.get("XResolution"))
                yres = _rational(page.tags.get("YResolution"))
                resunit_tag = page.tags.get("ResolutionUnit")
                resunit = int(resunit_tag.value) if resunit_tag is not None else None
                break

    meta = {"axes": axes,
        "imagej": ij_meta,
        "xres": xres,
        "yres": yres,
        "resunit": resunit}
    return arr, meta

In [36]:

def save_image(path, arr, meta):
    """Write array to path, preserving the original pixel size/spacing/unit metadata.
    Uses safe_imwrite so a failed write can never corrupt the existing file.
    """
    ij = meta["imagej"]
    axes = meta["axes"]
    md = {"axes": axes}

    for key in ("spacing", "unit", "finterval", "fps", "mode"):
        if key in ij:
            md[key] = ij[key]

    for ax, name in (("Z", "slices"), ("C", "channels"), ("T", "frames")):
        if ax in axes:
            md[name] = arr.shape[axes.index(ax)]

    kwargs = {"imagej": True, "metadata": md}
    if meta["xres"] and meta["yres"]:
        kwargs["resolution"] = (meta["xres"], meta["yres"])
    if meta["resunit"] is not None:
        kwargs["resolutionunit"] = meta["resunit"]

    safe_imwrite(path, arr, **kwargs)

In [37]:
class MaskCurator:
    """Napari GUI to curate the mask channel of a (Z, C, Y, X) tif.

    Saving strategy (optimised for very large stacks on slow / network drives):
      * While curating, only the mask channel (single ZYX channel) is autosaved
        to a small `<name>_MASK_AUTOSAVE.tif` on a BACKGROUND THREAD so the GUI
        stays responsive; the original file is never touched.
      * "Save final" commits: it folds the mask into the full stack, writes it
        over the original, then deletes the `_CURATED` / `_MASK_AUTOSAVE` sidecars.
      * Closing the window writes the full stack to a non-destructive
        `<name>_CURATED.tif` safety net, but only when there are unsaved edits.
      * On load it resumes from an existing `_CURATED`, and overlays a newer
        `_MASK_AUTOSAVE` onto the mask channel if one exists.
    All writes go through safe_imwrite (atomic temp-then-replace), and the mask is
    held as the smallest integer dtype that fits its labels (uint8/uint16) to prevent crashing/lagging
    """

    AUTOSAVE_SECONDS = 300
    REGION_GRID_THRESHOLD = 1024
    MAX_HISTORY = 10  # how many bulk operations can be undone

    def __init__(self, default_folder=None, brightfield_channel=0, mask_channel=2, cell_fluorescence_channel=1, perfused_fluorescence_channel=None):
        self.default_folder = Path(default_folder) if default_folder else None

        # Channel roles. brightfield + mask ("existing label") are required; the
        # cell / perfused fluorescence channels are optional (None = not shown).
        self.brightfield_channel = brightfield_channel
        self.mask_channel = mask_channel
        self.cell_fluorescence_channel = cell_fluorescence_channel
        self.perfused_fluorescence_channel = perfused_fluorescence_channel
        self.channel_count = None
        self.channel_widget = None

        self.viewer = None
        self.base_path = None      # original identity used for naming outputs
        self.image_stack = None    # loaded (Z, C, Y, X) array
        self.metadata = None
        self.mask_layer = None
        self.log_widget = None
        self.original_close_event = None
        self.autosave_timer = None
        self.save_worker = None    # kept referenced so it is not garbage-collected
        self.save_running = False
        self.has_unsaved_changes = False

        # Operation region controls (editing stays global; bulk ops can be scoped).
        self.region_widget = None
        self.operation_scope = "Full image"
        self.active_region = "R1"
        self.region_rows = 2
        self.region_cols = 2
        self.region_bounds = []
        self.region_outline_layer = None

        # Undo/redo history for the bulk operations (replace / interpolate /
        # fade out). Painting/erasing keep napari's own separate undo stack.
        self.undo_stack = []
        self.redo_stack = []
        self.undo_button = None
        self.redo_button = None
        self.history_widget = None

    def log(self, message):
        """Append a message to the GUI log panel (falls back to print)."""
        if self.log_widget is not None:
            current = self.log_widget.value
            self.log_widget.value = (current + "\n" + message) if current else message
        else:
            print(message)

    @staticmethod
    def contrast_limits(volume):
        """Min/max of a channel for a full-dynamic-range display stretch.

        Returns None when the volume is flat so napari uses its own defaults.
        """
        low, high = float(np.min(volume)), float(np.max(volume))
        return [low, high] if high > low else None

    def voxel_scale(self):
        """(z, y, x) voxel scale from the loaded metadata; missing values -> 1.0.

        Z spacing comes from the ImageJ `spacing` field; Y/X pixel sizes are the
        reciprocal of the TIFF Y/X resolution (stored as pixels-per-unit).
        """
        z = y = x = 1.0
        if self.metadata is not None:
            imagej_meta = self.metadata.get("imagej") or {}
            spacing = imagej_meta.get("spacing")
            if spacing:
                z = float(spacing)
            y_resolution = self.metadata.get("yres")
            if y_resolution:
                y = 1.0 / float(y_resolution)
            x_resolution = self.metadata.get("xres")
            if x_resolution:
                x = 1.0 / float(x_resolution)
        return (z, y, x)

    # ------------------------------------------------------------------ paths
    def curated_path(self):
        """Non-destructive full-stack safety net: `<base>_CURATED.tif`."""
        return self.base_path.with_name(self.base_path.stem + "_CURATED" + self.base_path.suffix)

    def mask_autosave_path(self):
        """Small mask-only autosave: `<base>_MASK_AUTOSAVE.tif`."""
        return self.base_path.with_name(self.base_path.stem + "_MASK_AUTOSAVE" + self.base_path.suffix)

    # --------------------------------------------------------------- regions
    @staticmethod
    def split_axis(length, part_count):
        """Split an axis into near-equal integer ranges [(start, end), ...]."""
        edges = np.linspace(0, int(length), int(part_count) + 1, dtype=int)
        return [(int(edges[i]), int(edges[i + 1])) for i in range(int(part_count))]

    def region_choices(self, widget=None):
        """Current region names for the dropdown (2x2 default before any load)."""
        names = [entry[0] for entry in self.region_bounds]
        return names or ["R1", "R2", "R3", "R4"]

    def configure_regions(self, mask_shape):
        """Build region bounds (4 for small images, 9 for large) and refresh the dropdown."""
        height, width = int(mask_shape[1]), int(mask_shape[2])
        if min(height, width) >= self.REGION_GRID_THRESHOLD:
            self.region_rows = self.region_cols = 3
        else:
            self.region_rows = self.region_cols = 2

        y_ranges = self.split_axis(height, self.region_rows)
        x_ranges = self.split_axis(width, self.region_cols)

        bounds = []
        region_index = 1
        for row in range(self.region_rows):
            for col in range(self.region_cols):
                y_start, y_end = y_ranges[row]
                x_start, x_end = x_ranges[col]
                bounds.append((f"R{region_index}", y_start, y_end, x_start, x_end))
                region_index += 1
        self.region_bounds = bounds

        valid_names = [entry[0] for entry in bounds]
        if self.active_region not in valid_names:
            self.active_region = valid_names[0]

        if self.region_widget is not None:
            self.region_widget.region.reset_choices()
            self.region_widget.region.value = self.active_region

    def get_region_bounds(self, region_name=None):
        """Return (name, y_start, y_end, x_start, x_end) for the requested region."""
        if not self.region_bounds:
            return None
        name = region_name if region_name is not None else self.active_region
        for entry in self.region_bounds:
            if entry[0] == name:
                return entry
        return self.region_bounds[0]

    def draw_region_outline(self):
        """Draw a thick outline around the active region while scoped mode is on."""
        if self.region_outline_layer is not None and self.viewer is not None:
            try:
                self.viewer.layers.remove(self.region_outline_layer)
            except Exception:
                pass
        self.region_outline_layer = None

        if self.viewer is None or self.mask_layer is None:
            return
        if self.operation_scope != "Selected region":
            return

        bounds = self.get_region_bounds()
        if bounds is None:
            return
        name, y_start, y_end, x_start, x_end = bounds

        # Rectangle in YX (2D overlay, visible regardless of z-slice).
        rectangle = np.array([
            [y_start, x_start],
            [y_start, max(x_start + 1, x_end - 1)],
            [max(y_start + 1, y_end - 1), max(x_start + 1, x_end - 1)],
            [max(y_start + 1, y_end - 1), x_start],
        ], dtype=float)

        # The image/mask layers are voxel-scaled, so the outline must share the
        # same YX scale or it will be offset and too small.
        yx_scale = self.voxel_scale()[1:]

        self.region_outline_layer = self.viewer.add_shapes(
            [rectangle], shape_type="rectangle", edge_color="yellow", edge_width=5,
            face_color="transparent", name="active region", ndim=2, scale=yx_scale)
        self.region_outline_layer.editable = False

    def set_operation_region(self, mode="Full image", region="R1"):
        """Update operation scope/region and refresh the visual outline."""
        self.operation_scope = str(mode)
        self.active_region = str(region)

        if self.region_widget is not None:
            self.region_widget.region.enabled = (self.operation_scope == "Selected region")

        self.draw_region_outline()

        if self.operation_scope == "Full image":
            self.log("[REGION] Operations set to full XY extent")
        else:
            bounds = self.get_region_bounds(self.active_region)
            if bounds is not None:
                name, y_start, y_end, x_start, x_end = bounds
                self.log(f"[REGION] Operations set to {name}: y={y_start}:{y_end}, x={x_start}:{x_end}")

    def get_operation_slices(self, mask_data):
        """Return (y_slice, x_slice, description) for the current operation scope."""
        height, width = int(mask_data.shape[-2]), int(mask_data.shape[-1])
        if self.operation_scope != "Selected region":
            return slice(0, height), slice(0, width), "full XY"

        bounds = self.get_region_bounds(self.active_region)
        if bounds is None:
            return slice(0, height), slice(0, width), "full XY"

        name, y_start, y_end, x_start, x_end = bounds
        return slice(y_start, y_end), slice(x_start, x_end), f"{name} (y={y_start}:{y_end}, x={x_start}:{x_end})"

    ################### undo/redo ###################
    def update_history_buttons(self):
        """Enable/disable the undo/redo buttons to match the history stacks."""
        if self.undo_button is not None:
            self.undo_button.enabled = bool(self.undo_stack)
        if self.redo_button is not None:
            self.redo_button.enabled = bool(self.redo_stack)

    def record_change(self, z_min, z_max, y_slice, x_slice, before_block):
        """Record one completed bulk edit so it can be undone/redone.

        Only the affected sub-volume is stored (a before/after pair) so memory
        stays bounded. A new edit clears the redo stack (standard undo semantics).
        """
        mask_data = np.asarray(self.mask_layer.data)
        after_block = mask_data[z_min:z_max + 1, y_slice, x_slice].copy()
        self.undo_stack.append((z_min, z_max, y_slice, x_slice, before_block, after_block))
        if len(self.undo_stack) > self.MAX_HISTORY:
            self.undo_stack.pop(0)
        self.redo_stack.clear()
        self.update_history_buttons()

    def apply_history_step(self, source_stack, target_stack, restore_index, label):
        """Move one entry between the undo/redo stacks, writing its stored block."""
        entry = source_stack.pop()
        z_min, z_max, y_slice, x_slice = entry[0], entry[1], entry[2], entry[3]
        mask_data = np.asarray(self.mask_layer.data)
        mask_data[z_min:z_max + 1, y_slice, x_slice] = entry[restore_index]
        self.mask_layer.data = mask_data
        self.mask_layer.refresh()
        target_stack.append(entry)
        self.has_unsaved_changes = True
        self.log(f"[{label}] operation on z={z_min}:{z_max}")
        self.update_history_buttons()
        self.save_mask_async()  # secure the result

    def undo(self):
        """Revert the most recent bulk operation (restores its 'before' block)."""
        if self.mask_layer is None or not self.undo_stack:
            self.log("[UNDO] nothing to undo.")
            return
        self.apply_history_step(self.undo_stack, self.redo_stack, restore_index=4, label="UNDO")

    def redo(self):
        """Re-apply the most recently undone operation (restores its 'after' block)."""
        if self.mask_layer is None or not self.redo_stack:
            self.log("[REDO] nothing to redo.")
            return
        self.apply_history_step(self.redo_stack, self.undo_stack, restore_index=5, label="REDO")

    ################### GUI ###################
    def start(self):
        from qtpy.QtCore import QTimer

        self.viewer = napari.Viewer(title="Mask curation")

        self.load_widget = magicgui(
            self.load_image,
            image_path={"label": "Image", "mode": "r",
                        "filter": "TIFF (*.tif *.tiff)"},
            call_button="Load image",
        )
        if self.default_folder and self.default_folder.exists():
            self.load_widget.image_path.value = self.default_folder

        # Assign which channel is which role. brightfield + existing label are
        # required; cell / perfused fluorescence are optional ("None" = hidden).
        self.channel_widget = magicgui(
            self.apply_channel_assignment,
            call_button="Apply channel assignment",
            brightfield={"label": "Brightfield",
                         "choices": lambda widget=None: self.channel_choices(include_none=False)},
            existing_label={"label": "Existing label",
                            "choices": lambda widget=None: self.channel_choices(include_none=False)},
            cell_fluorescence={"label": "Fluorescence (cell)",
                               "choices": lambda widget=None: self.channel_choices(include_none=True)},
            perfused_fluorescence={"label": "Fluorescence (perfused)",
                                   "choices": lambda widget=None: self.channel_choices(include_none=True)},
        )
        self.channel_widget.brightfield.value = self.brightfield_channel
        self.channel_widget.existing_label.value = self.mask_channel
        self.channel_widget.cell_fluorescence.value = self.cell_fluorescence_channel
        self.channel_widget.perfused_fluorescence.value = self.perfused_fluorescence_channel

        # Size the channel dropdowns to the file's channel count as soon as it is
        # picked, so the allowed range is right before "Load image" is clicked.
        self.load_widget.image_path.changed.connect(self.on_image_path_changed)
        self.on_image_path_changed()

        self.region_widget = magicgui(
            self.set_operation_region,
            auto_call=True,
            mode={"label": "Operation scope", "choices": ["Full image", "Selected region"]},
            region={"label": "Region", "choices": self.region_choices},
        )

        # Replace a z-slice (or an inclusive range) of the MASK with another slice.
        self.replace_widget = magicgui(
            self.replace_slices,
            replace_z={"label": "Replace z (e.g. 16 or 1,3)"},
            source_z={"label": "with mask from z"},
            call_button="Replace slices",
        )

        # Interpolate the MASK between two z-slices (signed-distance morphing).
        self.interpolate_widget = magicgui(
            self.interpolate_slices,
            z_min={"label": "z min"},
            z_max={"label": "z max"},
            call_button="Interpolate",
        )

        # Fade out (shrink to empty) the MASK from one z-slice to another.
        self.fade_widget = magicgui(
            self.fade_out_slices,
            z_start={"label": "fade out from z"},
            z_end={"label": "to z"},
            call_button="Fade out",
        )

        # Undo / redo for the bulk operations above (separate from napari's
        # built-in paint/erase undo).
        self.undo_button = PushButton(text="Undo last operation")
        self.redo_button = PushButton(text="Redo")
        self.undo_button.clicked.connect(self.undo)
        self.redo_button.clicked.connect(self.redo)
        self.history_widget = Container(
            widgets=[self.undo_button, self.redo_button],
            layout="horizontal", labels=False,
        )

        # Manual FINAL save = COMMIT the full stack over the original.
        self.save_widget = magicgui(self.save_full,
                                    call_button="Save final (overwrite original)")

        self.log_widget = TextEdit(value="", label="Log")
        try:
            self.log_widget.native.setReadOnly(True)
        except Exception:
            pass
        self.log_widget.min_height = 120
        self.log_widget.max_height = 300

        self.viewer.window.add_dock_widget(self.load_widget, area="right",
                                           name="Select image")
        self.viewer.window.add_dock_widget(self.channel_widget, area="right",
                                           name="Channel assignment")
        self.viewer.window.add_dock_widget(self.region_widget, area="right",
                                           name="Operation region")
        self.viewer.window.add_dock_widget(self.replace_widget, area="right",
                                           name="Replace slices")
        self.viewer.window.add_dock_widget(self.interpolate_widget, area="right",
                                           name="Interpolate between")
        self.viewer.window.add_dock_widget(self.fade_widget, area="right",
                                           name="Fade out")
        self.viewer.window.add_dock_widget(self.history_widget, area="right",
                                           name="Undo / redo operations")
        self.viewer.window.add_dock_widget(self.save_widget, area="right",
                                           name="Save")
        self.viewer.window.add_dock_widget(self.log_widget, area="right",
                                           name="Log")

        # Autosave (mask only, background thread) on a timer.
        self.autosave_timer = QTimer()
        self.autosave_timer.setInterval(self.AUTOSAVE_SECONDS * 1000)
        self.autosave_timer.timeout.connect(self.autosave)
        self.autosave_timer.start()

        # Finalise (full-stack save) when the window is closed.
        qt_window = self.viewer.window._qt_window
        self.original_close_event = qt_window.closeEvent
        qt_window.closeEvent = self.on_close

        # Start the region UI and undo/redo buttons in a valid state.
        self.set_operation_region(self.operation_scope, self.active_region)
        self.update_history_buttons()

    ################### load ###################
    def load_image(self, image_path=Path()):
        image_path = Path(image_path)
        if not image_path.is_file():
            self.log("[WARN] Please select a tif file.")
            return

        # Identity/base name (strip a _CURATED suffix if the user picked one).
        stem = image_path.stem
        if stem.endswith("_CURATED"):
            self.base_path = image_path.with_name(stem[: -len("_CURATED")] + image_path.suffix)
        else:
            self.base_path = image_path

        # Prefer an existing full _CURATED as the source of pixel data.
        curated_file = self.curated_path()
        source_path = curated_file if curated_file.is_file() else self.base_path
        if source_path == curated_file:
            self.log(f"[RESUME] Loading full progress from {curated_file.name}")

        try:
            image_stack, metadata = read_image_and_meta(source_path)
        except Exception as error:
            self.log(f"[ERROR] Could not read {source_path.name}: {type(error).__name__}: {error}")
            return

        self.image_stack, self.metadata = image_stack, metadata

        # Refresh the channel dropdowns to match this stack, then read back the
        # (possibly adjusted) role selections before anything uses them.
        self.channel_count = int(self.image_stack.shape[1]) if self.image_stack.ndim >= 2 else 1
        if self.channel_widget is not None:
            self.channel_widget.reset_choices()
        self.read_channel_assignment()

        # If a mask-only autosave is newer than the ORIGINAL file, overlay it onto
        # the mask channel (compared against base_path so a _CURATED sidecar can't
        # hide a newer autosave).
        autosave_file = self.mask_autosave_path()
        original_mtime = (self.base_path.stat().st_mtime
                          if self.base_path.is_file() else source_path.stat().st_mtime)
        if autosave_file.is_file() and autosave_file.stat().st_mtime > original_mtime:
            try:
                mask_data = tifffile.imread(autosave_file)
                expected_shape = self.image_stack[:, self.mask_channel].shape
                if mask_data.shape == expected_shape:
                    # Direct assignment casts in place -> no full-size temp copy.
                    self.image_stack[:, self.mask_channel] = mask_data
                    self.log(f"[RESUME] Applied newer mask autosave {autosave_file.name}")
                else:
                    self.log(f"[WARN] Mask autosave shape {mask_data.shape} != "
                             f"expected {expected_shape}; ignored.")
                del mask_data
            except Exception as error:
                self.log(f"[WARN] Could not read mask autosave: {error}")

        if not self.build_layers():
            return

        self.viewer.title = self.base_path.name
        self.has_unsaved_changes = False
        self.undo_stack = []       # fresh image -> clear operation history
        self.redo_stack = []
        self.update_history_buttons()
        self.log(f"[OK] Loaded {source_path.name}\n"
                 f"  mask autosave -> {self.mask_autosave_path().name}\n"
                 f"  save final    -> overwrites {self.base_path.name}")

    ################### channels ###################
    @staticmethod
    def peek_channel_count(path):
        """Read the channel count from a tif's header WITHOUT loading the pixels."""
        try:
            with tifffile.TiffFile(path) as tif:
                series = max(tif.series, key=lambda s: s.size)
                axes, shape = series.axes, series.shape
            if "C" in axes:
                return int(shape[axes.index("C")])
            if len(shape) >= 4:      # fall back to the (Z, C, Y, X) convention
                return int(shape[1])
        except Exception:
            return None
        return None

    def on_image_path_changed(self, *args):
        """Resize the channel dropdowns to match the currently selected file."""
        if self.channel_widget is None:
            return
        path = Path(self.load_widget.image_path.value)
        if not path.is_file():
            return
        channel_count = self.peek_channel_count(path)
        if channel_count and channel_count != self.channel_count:
            self.channel_count = channel_count
            self.channel_widget.reset_choices()
            self.log(f"[CHANNELS] {path.name} has {channel_count} channels; dropdowns updated.")

    def channel_choices(self, include_none):
        """Channel-index options for a role dropdown, optionally with a "None" entry."""
        count = self.channel_count or 3
        options = [(str(index), index) for index in range(count)]
        return ([("None", None)] + options) if include_none else options

    def read_channel_assignment(self):
        """Pull the current role -> channel selections from the GUI dropdowns."""
        if self.channel_widget is None:
            return
        self.brightfield_channel = self.channel_widget.brightfield.value
        self.mask_channel = self.channel_widget.existing_label.value
        self.cell_fluorescence_channel = self.channel_widget.cell_fluorescence.value
        self.perfused_fluorescence_channel = self.channel_widget.perfused_fluorescence.value

    def apply_channel_assignment(self, brightfield=0, existing_label=2,
                                 cell_fluorescence=None, perfused_fluorescence=None):
        """Assign which channel plays each role and (re)build the display layers.

        `brightfield` and `existing_label` are required; the cell / perfused
        fluorescence channels are optional (None -> that layer is not shown).
        """
        # Preserve any in-progress mask edits before rebuilding the layers.
        if self.image_stack is not None and self.mask_layer is not None and self.has_unsaved_changes:
            self.sync_mask_into_stack()

        self.brightfield_channel = brightfield
        self.mask_channel = existing_label
        self.cell_fluorescence_channel = cell_fluorescence
        self.perfused_fluorescence_channel = perfused_fluorescence

        if self.image_stack is None:
            self.log("[CHANNELS] Assignment saved; it applies when you load an image.")
            return
        self.build_layers()

    def build_layers(self):
        """(Re)build the napari layers from `self.image_stack` for the current roles.

        Returns True on success, False if a required channel is invalid. Each
        intensity layer is min/max stretched for good dynamic range.
        """
        if self.image_stack is None or self.viewer is None:
            return False

        channel_count = int(self.image_stack.shape[1])

        def channel_in_range(channel):
            return channel is not None and 0 <= int(channel) < channel_count

        if not channel_in_range(self.brightfield_channel):
            self.log(f"[ERROR] Brightfield channel {self.brightfield_channel} is "
                     f"out of range (image has {channel_count} channels).")
            return False
        if not channel_in_range(self.mask_channel):
            self.log(f"[ERROR] Existing-label channel {self.mask_channel} is "
                     f"out of range (image has {channel_count} channels).")
            return False

        scale = self.voxel_scale()
        self.log(f"[OK] Using voxel scale (z, y, x) = {scale}")

        # Replace any layers from a previously loaded image / assignment.
        self.region_outline_layer = None
        self.viewer.layers.clear()

        brightfield = self.image_stack[:, self.brightfield_channel, :, :]
        self.viewer.add_image(brightfield, name="brightfield",
                              colormap="gray", blending="additive", scale=scale,
                              contrast_limits=self.contrast_limits(brightfield))

        if channel_in_range(self.cell_fluorescence_channel):
            cell = self.image_stack[:, self.cell_fluorescence_channel, :, :]
            self.viewer.add_image(cell, name="fluorescence (cell)",
                                  colormap="green", blending="additive", scale=scale,
                                  contrast_limits=self.contrast_limits(cell))

        if channel_in_range(self.perfused_fluorescence_channel):
            perfused = self.image_stack[:, self.perfused_fluorescence_channel, :, :]
            self.viewer.add_image(perfused, name="fluorescence (perfused)",
                                  colormap="magenta", blending="additive", scale=scale,
                                  contrast_limits=self.contrast_limits(perfused))

        mask = self.image_stack[:, self.mask_channel, :, :]
        # Hold the mask as the smallest integer dtype that fits its labels to keep
        # the RAM footprint low on these large XY stacks.
        label_dtype = np.uint8 if int(mask.max()) <= 255 else np.uint16
        self.mask_layer = self.viewer.add_labels(mask.astype(label_dtype),
                                                 name="mask", scale=scale)
        self.mask_layer.events.paint.connect(self.mark_dirty)
        self.mask_layer.events.data.connect(self.mark_dirty)

        self.configure_regions(mask.shape)
        self.set_operation_region(self.operation_scope, self.active_region)

        self.viewer.layers.selection.active = self.mask_layer
        self.log(f"[OK] Channels -> brightfield={self.brightfield_channel}, "
                 f"existing label={self.mask_channel}, "
                 f"cell={self.cell_fluorescence_channel}, perfused={self.perfused_fluorescence_channel}")
        return True

    @staticmethod
    def parse_z_range(text, slice_count):
        """'16' -> [16];  '1,3' or '1-3' -> [1,2,3] (inclusive). None if invalid."""
        cleaned = str(text).strip().strip("{}[]()")
        for separator in ("-", " ", ";"):
            cleaned = cleaned.replace(separator, ",")
        parts = [part for part in cleaned.split(",") if part != ""]
        if not parts:
            return None
        try:
            numbers = [int(part) for part in parts]
        except ValueError:
            return None
        z_min, z_max = (numbers[0], numbers[0]) if len(numbers) == 1 else (numbers[0], numbers[1])
        if z_min > z_max:
            z_min, z_max = z_max, z_min
        if z_min < 0 or z_max >= slice_count:
            return None
        return list(range(z_min, z_max + 1))

    @staticmethod
    def parse_single_z(text, slice_count):
        """Parse a single slice index; None if invalid or out of range."""
        cleaned = str(text).strip().strip("{}[]()")
        try:
            z = int(cleaned)
        except ValueError:
            return None
        return z if 0 <= z < slice_count else None

    def replace_slices(self, replace_z: str = "", source_z: str = ""):
        """Copy the MASK from `source_z` into every slice in `replace_z`.

        Operates ONLY on the mask layer; the intensity layers are never touched.
        """
        if self.mask_layer is None:
            self.log("[WARN] Load an image before replacing slices.")
            return

        mask_data = np.asarray(self.mask_layer.data)
        slice_count = mask_data.shape[0]

        target_slices = self.parse_z_range(replace_z, slice_count)
        if target_slices is None:
            self.log(f"[ERROR] 'Replace z' = {replace_z!r} is invalid "
                     f"(use e.g. 16 or 1,3; valid range 0..{slice_count - 1}).")
            return
        source = self.parse_single_z(source_z, slice_count)
        if source is None:
            self.log(f"[ERROR] 'with mask from z' = {source_z!r} is invalid "
                     f"(use a single slice 0..{slice_count - 1}).")
            return

        y_slice, x_slice, scope_description = self.get_operation_slices(mask_data)
        z_min, z_max = min(target_slices), max(target_slices)
        before_block = mask_data[z_min:z_max + 1, y_slice, x_slice].copy()
        source_plane = mask_data[source, y_slice, x_slice].copy()
        for z in target_slices:
            mask_data[z, y_slice, x_slice] = source_plane
        self.mask_layer.data = mask_data
        self.mask_layer.refresh()
        self.record_change(z_min, z_max, y_slice, x_slice, before_block)

        self.has_unsaved_changes = True
        if len(target_slices) == 1:
            self.log(f"[OK] Successfully replaced mask slice {target_slices[0]} "
                     f"with slice {source} in {scope_description}")
        else:
            self.log(f"[OK] Successfully replaced mask slices "
                     f"{target_slices[0]}-{target_slices[-1]} with slice {source} in {scope_description}")
        self.save_mask_async()

    @staticmethod
    def signed_distance(binary_mask):
        """Signed distance transform: positive inside the mask, negative outside."""
        binary_mask = binary_mask.astype(bool)
        return distance_transform_edt(binary_mask) - distance_transform_edt(~binary_mask)

    def interpolate_slices(self, z_min: str = "", z_max: str = ""):
        """Interpolate the MASK between slices `z_min` and `z_max` (inclusive) by
        signed-distance morphing. The two endpoint slices are kept exactly; every
        slice in between is filled in. Operates ONLY on the mask layer.
        """
        if self.mask_layer is None:
            self.log("[WARN] Load an image before interpolating.")
            return

        mask_data = np.asarray(self.mask_layer.data)
        slice_count = mask_data.shape[0]

        start = self.parse_single_z(z_min, slice_count)
        if start is None:
            self.log(f"[ERROR] 'z min' = {z_min!r} is invalid "
                     f"(use a single slice 0..{slice_count - 1}).")
            return
        end = self.parse_single_z(z_max, slice_count)
        if end is None:
            self.log(f"[ERROR] 'z max' = {z_max!r} is invalid "
                     f"(use a single slice 0..{slice_count - 1}).")
            return
        if start == end:
            self.log("[ERROR] z min and z max must be different slices.")
            return
        if start > end:
            start, end = end, start

        y_slice, x_slice, scope_description = self.get_operation_slices(mask_data)
        before_block = mask_data[start:end + 1, y_slice, x_slice].copy()
        start_mask = mask_data[start, y_slice, x_slice]
        end_mask = mask_data[end, y_slice, x_slice]

        # Blend the two signed-distance fields, reproducing the endpoints exactly,
        # then write the whole span back in one assignment.
        start_sdf = self.signed_distance(start_mask)
        end_sdf = self.signed_distance(end_mask)
        span = end - start + 1
        interpolated = np.zeros((span, *start_mask.shape), dtype=mask_data.dtype)
        for offset in range(span):
            fraction = offset / (span - 1)
            interpolated[offset] = ((1.0 - fraction) * start_sdf + fraction * end_sdf) >= 0
        interpolated[0] = start_mask.astype(bool)
        interpolated[-1] = end_mask.astype(bool)

        label_value = int(max(int(start_mask.max()), int(end_mask.max()))) or 1
        mask_data[start:end + 1, y_slice, x_slice] = interpolated * label_value
        self.mask_layer.data = mask_data
        self.mask_layer.refresh()
        self.record_change(start, end, y_slice, x_slice, before_block)

        self.has_unsaved_changes = True
        self.log(f"[OK] Successfully interpolated mask slices {start}-{end} "
                 f"(endpoints {start} and {end} kept) in {scope_description}")
        self.save_mask_async()

    def fade_out_slices(self, z_start: str = "", z_end: str = ""):
        """Fade the MASK out from `z_start` (kept as-is) to `z_end` (set empty),
        progressively shrinking the shape in between via a distance-transform
        threshold. Operates ONLY on the mask layer; direction is honoured.
        """
        if self.mask_layer is None:
            self.log("[WARN] Load an image before fading out.")
            return

        mask_data = np.asarray(self.mask_layer.data)
        slice_count = mask_data.shape[0]

        start = self.parse_single_z(z_start, slice_count)
        if start is None:
            self.log(f"[ERROR] 'fade out from z' = {z_start!r} is invalid "
                     f"(use a single slice 0..{slice_count - 1}).")
            return
        end = self.parse_single_z(z_end, slice_count)
        if end is None:
            self.log(f"[ERROR] 'to z' = {z_end!r} is invalid "
                     f"(use a single slice 0..{slice_count - 1}).")
            return
        if start == end:
            self.log("[ERROR] fade-out start and end must be different slices.")
            return

        y_slice, x_slice, scope_description = self.get_operation_slices(mask_data)
        z_min, z_max = min(start, end), max(start, end)
        before_block = mask_data[z_min:z_max + 1, y_slice, x_slice].copy()
        source_mask = mask_data[start, y_slice, x_slice].astype(bool)
        label_value = int(mask_data[start, y_slice, x_slice].max()) or 1

        # Distance of each foreground pixel from the nearest background pixel
        distance = distance_transform_edt(source_mask)
        max_distance = distance.max()
        fade_length = abs(end - start) + 1
        step = 1 if end > start else -1
        for offset, z in enumerate(range(start, end + step, step)):
            if offset == 0:
                plane = source_mask
            elif offset == fade_length - 1:
                plane = np.zeros_like(source_mask)
            else:
                plane = distance > (offset / (fade_length - 1)) * max_distance
            mask_data[z, y_slice, x_slice] = plane.astype(mask_data.dtype) * label_value

        self.mask_layer.data = mask_data
        self.mask_layer.refresh()
        self.record_change(z_min, z_max, y_slice, x_slice, before_block)

        self.has_unsaved_changes = True
        self.log(f"[OK] Successfully faded out mask from z={start} to z={end} "
                 f"(z={end} set empty) in {scope_description}")
        self.save_mask_async()

    ################### save ###################
    def mark_dirty(self, event=None):
        self.has_unsaved_changes = True

    def sync_mask_into_stack(self):
        # Direct assignment casts element-wise into the existing buffer, avoiding
        # a second full-size copy that could exhaust RAM on big stacks.
        self.image_stack[:, self.mask_channel, :, :] = self.mask_layer.data

    def on_save_finished(self):
        """Runs on the GUI thread when any background save worker ends."""
        self.save_running = False
        self.save_worker = None

    def save_mask_async(self):
        """Autosave ONLY the mask channel (ZYX) on a background thread."""
        if self.mask_layer is None or self.base_path is None:
            return
        if self.save_running:
            self.log("[autosave] previous save still running; will retry next tick.")
            return

        # Snapshot on the GUI thread so the worker has a stable, private copy;
        # keep the layer's compact dtype rather than upcasting.
        mask_snapshot = np.array(self.mask_layer.data)
        output_path = self.mask_autosave_path()

        # ImageJ kwargs for the mask-only (ZYX) write.
        write_metadata = {"axes": "ZYX"}
        imagej_meta = self.metadata["imagej"]
        for key in ("spacing", "unit"):
            if key in imagej_meta:
                write_metadata[key] = imagej_meta[key]
        write_kwargs = {"imagej": True, "metadata": write_metadata}
        if self.metadata["xres"] and self.metadata["yres"]:
            write_kwargs["resolution"] = (self.metadata["xres"], self.metadata["yres"])
        if self.metadata["resunit"] is not None:
            write_kwargs["resolutionunit"] = self.metadata["resunit"]

        self.has_unsaved_changes = False  # captured in snapshot; worker re-flags on failure

        @thread_worker
        def write_mask():
            safe_imwrite(output_path, mask_snapshot, **write_kwargs)
            return output_path.name

        def on_success(name):
            self.log(f"[autosave] save completed -> {name}")

        def on_error(error):
            self.has_unsaved_changes = True  # force a retry on the next tick
            self.log(f"[autosave][ERROR] {type(error).__name__}: {error}")

        worker = write_mask()
        worker.returned.connect(on_success)
        worker.errored.connect(on_error)
        worker.finished.connect(self.on_save_finished)
        self.save_worker = worker
        self.save_running = True
        worker.start()
        self.log(f"[autosave] writing mask in background -> {output_path.name} ...")

    def autosave(self):
        if self.has_unsaved_changes and self.mask_layer is not None and self.base_path is not None:
            self.save_mask_async()

    def save_full(self):
        """FINAL save / COMMIT: fold the mask into the full stack, write it OVER
        the original, then delete the now-obsolete _CURATED / _MASK_AUTOSAVE
        sidecars. Runs on a background thread.
        """
        if self.mask_layer is None or self.image_stack is None:
            self.log("[WARN] Load an image before saving.")
            return
        if self.save_running:
            self.log("[SAVE] a save is already in progress; try again shortly.")
            return

        self.sync_mask_into_stack()       # fold current mask in on the GUI thread
        self.has_unsaved_changes = False  # snapshot taken; a later edit re-flags via paint
        output_path = self.base_path      # COMMIT over the original file
        image_stack, metadata = self.image_stack, self.metadata
        curated_file = self.curated_path()
        autosave_file = self.mask_autosave_path()

        @thread_worker
        def write_full():
            save_image(output_path, image_stack, metadata)
            # original now holds the curated result -> remove obsolete sidecars
            for sidecar in (curated_file, autosave_file):
                try:
                    if sidecar != output_path and sidecar.is_file():
                        sidecar.unlink()
                except Exception:
                    pass
            return output_path.name

        def on_success(name):
            self.log(f"[SAVE] save completed -> {name} "
                     f"(original overwritten; sidecars removed)")

        def on_error(error):
            self.has_unsaved_changes = True  # commit failed -> keep so it can be re-saved
            self.log(f"[SAVE][ERROR] {type(error).__name__}: {error}; "
                     f"original + sidecars left intact.")

        worker = write_full()
        worker.returned.connect(on_success)
        worker.errored.connect(on_error)
        worker.finished.connect(self.on_save_finished)
        self.save_worker = worker
        self.save_running = True
        worker.start()
        self.log(f"[SAVE] writing full stack OVER original -> {output_path.name} "
                 f"(this can take a while)...")

    def on_close(self, event):
        """On close: if there are unsaved mask edits, write the full stack
        SYNCHRONOUSLY to the non-destructive _CURATED safety net (so the process
        cannot exit mid-write and the original is untouched). Skip if unchanged.
        """
        if self.autosave_timer is not None:
            self.autosave_timer.stop()

        if not self.has_unsaved_changes:
            self.log("[CLOSE] no unsaved changes; nothing to save.")
            self.original_close_event(event)
            return

        if self.mask_layer is not None and self.image_stack is not None:
            try:
                self.sync_mask_into_stack()
                output_path = self.curated_path()
                self.log(f"[CLOSE] saving progress -> {output_path.name} ...")
                save_image(output_path, self.image_stack, self.metadata)
                self.has_unsaved_changes = False
                self.log(f"[CLOSE] save completed -> {output_path.name}")
            except Exception as error:
                self.log(f"[CLOSE][ERROR] {type(error).__name__}: {error}; "
                         f"mask autosave kept.")
        self.original_close_event(event)


In [38]:
# Default folder the file explorer opens in (you can change to wherever your images live, or just navigate to the folder each time from the GUI).
masks_folder = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate")

In [39]:
curator = MaskCurator(default_folder=masks_folder)
curator.start()

In [40]:
# input_path = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate\CART_day7_FL32_FL32_ARi2_Merged.tif")
# image = tifffile.imread(input_path)
# print(image.shape)

In [41]:

# def signed_distance(mask: np.ndarray) -> np.ndarray:
#     """
#     Positive inside the mask, negative outside.
#     """
#     mask = mask.astype(bool)
#     distance_inside = distance_transform_edt(mask)
#     distance_outside = distance_transform_edt(~mask)
#     return distance_inside - distance_outside



# z_end = 20
# z_start = 2
# mask_start = image[z_start,2,:,:]
# mask_end = image[z_end,2,:,:]
# sdf_start = signed_distance(mask_start)
# sdf_end = signed_distance(mask_end)

# number_of_slices = z_end - z_start + 1
# interpolated = np.zeros((number_of_slices, *mask_start.shape), dtype=np.uint8)

# for index in range(number_of_slices):
#     t = index / (number_of_slices - 1)
#     interpolated_sdf = ((1.0 - t) * sdf_start + t * sdf_end)
#     interpolated[index] = (interpolated_sdf >= 0).astype(np.uint8)

# # Ensure the supplied endpoint masks are reproduced exactly
# interpolated[0] = mask_start.astype(np.uint8)
# interpolated[-1] = mask_end.astype(np.uint8)

# fig, ax = plt.subplots(ncols = 3)
# ax[0].imshow(mask_start)
# ax[1].imshow(image[11,2,:,:])
# ax[2].imshow(mask_end)
# plt.show()

# fig, ax = plt.subplots(ncols = 3)
# ax[0].imshow(interpolated[0])
# ax[1].imshow(interpolated[-1])
# ax[2].imshow(interpolated[9])


In [42]:
# viewer = napari.Viewer()
# viewer.add_labels(image[:,2,:,:])
# viewer.add_labels(interpolated)

In [43]:

# def interpolate_binary_masks(mask_start: np.ndarray, mask_end: np.ndarray, z_start: int, z_end: int,) -> np.ndarray:
#     """
#     Interpolate binary masks between two known z-slices.

#     Returns an array with shape:
#         (z_end - z_start + 1, height, width)

#     The first and last slices are the supplied masks.
#     """
#     if mask_start.shape != mask_end.shape:
#         raise ValueError("The two masks must have the same shape.")

#     if z_end <= z_start:
#         raise ValueError("z_end must be greater than z_start.")

#     sdf_start = signed_distance(mask_start)
#     sdf_end = signed_distance(mask_end)

#     number_of_slices = z_end - z_start + 1
#     interpolated = np.zeros((number_of_slices, *mask_start.shape), dtype=np.uint8)

#     for index in range(number_of_slices):
#         t = index / (number_of_slices - 1)
#         interpolated_sdf = ((1.0 - t) * sdf_start + t * sdf_end)
#         interpolated[index] = (interpolated_sdf >= 0).astype(np.uint8)

#     # Ensure the supplied endpoint masks are reproduced exactly
#     interpolated[0] = mask_start.astype(np.uint8)
#     interpolated[-1] = mask_end.astype(np.uint8)

#     return interpolated

In [44]:
# for z in range(18, image.shape[0]):
#     image[z,2,:,:] = image[17,2,:,:]

In [45]:
# import numpy as np
# from scipy.ndimage import distance_transform_edt


# def shrink_mask_to_empty(
#     mask: np.ndarray,
#     number_of_slices: int,
# ) -> np.ndarray:
#     """
#     Generate slices that progressively shrink a binary mask to an empty mask.

#     Parameters
#     ----------
#     mask
#         2D binary mask.
#     number_of_slices
#         Total number of slices, including the original and empty endpoints.

#     Returns
#     -------
#     np.ndarray
#         Shape: (number_of_slices, height, width)
#     """
#     if number_of_slices < 2:
#         raise ValueError("number_of_slices must be at least 2.")

#     mask = mask.astype(bool)

#     output = np.zeros(
#         (number_of_slices, *mask.shape),
#         dtype=np.uint8,
#     )

#     if not mask.any():
#         return output

#     # Distance of each foreground pixel from the nearest background pixel
#     distance = distance_transform_edt(mask)
#     maximum_distance = distance.max()

#     for index in range(number_of_slices):
#         t = index / (number_of_slices - 1)

#         if index == 0:
#             output[index] = mask
#         elif index == number_of_slices - 1:
#             output[index] = 0
#         else:
#             threshold = t * maximum_distance
#             output[index] = (distance > threshold).astype(np.uint8)

#     return output

# In Case of Memory Error

In [46]:
# import tifffile, numpy as np

# layer = curator.mask_layer.data          # int32 array already in RAM (no copy made)
# mx = int(np.asarray(layer).max())
# dtype = np.uint8 if mx <= 255 else (np.uint16 if mx <= 65535 else np.int32)

# rescue = curator.base_path.with_name(curator.base_path.stem + "_MASK_RESCUE.tif")
# with tifffile.TiffWriter(rescue) as tif:
#     for z in range(layer.shape[0]):
#         plane = np.ascontiguousarray(layer[z]).astype(dtype)  # ~31 MiB, one slice only
#         tif.write(plane, contiguous=True)
#         del plane
# print("rescued mask ->", rescue)

In [47]:
# # Merge a 3-channel image with a mask-only autosave, writing the result to a NEW file.
# image_path = Path(r"z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate\Bel_mCh_BF_VASCUMAP_FL33_ARi1_Merged.tif")
# mask_path = Path(r"z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate\Bel_mCh_BF_VASCUMAP_FL33_ARi1_Merged_MASK_AUTOSAVE.tif")
# mask_channel = 2  # channel index the curated mask lives in (matches MaskCurator default)

# arr, meta = read_image_and_meta(image_path)   # (Z, C, Y, X)
# mask_arr = tifffile.imread(mask_path)          # mask-only (Z, Y, X)

# expected = arr[:, mask_channel].shape
# if mask_arr.shape != expected:
#     raise ValueError(f"Mask shape {mask_arr.shape} != expected {expected} for channel {mask_channel}.")

# # Overlay the autosave mask onto the mask channel (cast in place, no full-size copy).
# arr[:, mask_channel] = mask_arr
# del mask_arr

# # New output path (never overwrites the originals).
# out_path = image_path.with_name(image_path.stem + "_MERGED_WITH_AUTOSAVE" + image_path.suffix)
# save_image(out_path, arr, meta)
# print("merged ->", out_path)
